In [ ]:
# ================== Install dependencies in Colab ==================
!pip install -q langchain langchain-openai langchain-groq langchain_community langchain-huggingface faiss-cpu pandas openpyxl sentence-transformers
#!pip install -q langchain langchain_openai faiss-cpu openpyxl pandas langchain-groq
#!pip install -q langchain langchain-community langchain-groq


import pandas as pd
import os
#from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS   # ✅ updated import

# import pandas as pd
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_groq import ChatGroq
# from langchain.prompts import ChatPromptTemplate
# from langchain.vectorstores import FAISS
# import os

# ================== STEP 1: Load dataset ==================
products = pd.read_excel("/content/Product_Email_Dataset.xlsx", sheet_name="Products")
emails = pd.read_excel("/content/Product_Email_Dataset.xlsx", sheet_name="Client Emails")

print("Products:")
print(products.head())

# ================== STEP 2: Create Vectorstore of product descriptions ==================
#embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Convert products into documents for retrieval
product_texts = [
    f"Product: {row['Name']}. {row['Description']}. Season: {row['Seasons']}. Price: {row['Price']}. Stock: {row['Stock']}"
    for _, row in products.iterrows()
]

vectorstore = FAISS.from_texts(product_texts, embeddings)

# ================== STEP 3: LLM initialization ==================

# os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
# llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt_template = ChatPromptTemplate.from_template(
    """
    You are a helpful customer support assistant.

    Customer email:
    {email_text}

    Retrieved products from database:
    {retrieved_products}

    Instructions:
    - ONLY use the retrieved products to answer.
    - DO NOT make up any products.
    - If no relevant product is found, clearly say: "Sorry, this product is not available in our database."
    - If product exists, provide correct price and stock.
    - If email is unrelated, give a general polite response.
    """
)

# ================== STEP 4: Response generation with retrieval ==================
def generate_response(email_subject, email_body, top_k=3):
    email_text = (email_subject or "") + " " + (email_body or "")

    # # Exact match check (very important)
    # for _, row in products.iterrows():
    #     product_name = str(row['Name']).lower()
    #     if product_name in email_text.lower():
    #         return f"""
    # Subject: Re: Product Inquiry

    # Dear Customer,

    # The product "{row['Name']}" is available.

    # Price: ${row['Price']}
    # Stock: {row['Stock']} units

    # Let us know if you'd like to place an order.

    # Best regards,
    # Customer Support
    # """

    # Retrieve most relevant products
    docs_with_scores = vectorstore.similarity_search_with_score(email_text, k=top_k)

    # Filter relevant results only
    filtered_docs = [doc for doc, score in docs_with_scores if score < 1.0]

    if not filtered_docs:
        retrieved_products = "No relevant product found"
    else:
        retrieved_products = "\n".join([doc.page_content for doc in filtered_docs])

    chain = prompt_template | llm
    response = chain.invoke({"email_text": email_text, "retrieved_products": retrieved_products})
    return response.content

# ================== STEP 5: Test ==================
sample = emails.sample(5)
for _, row in sample.iterrows():
    print("\nEmail:", row["email_subject"], row["email_body"])
    print("Response:", generate_response(row["email_subject"], row["email_body"]))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
Products:
  Product ID                 Name         Category  \
0       P001       Smart Widget X    Smart Devices   
1       P002     Smart Thermostat  Home Automation   
2       P003      Pro Gadget Pack    Smart Devices   
3       P004  Eco-Friendly Widget    Smart Devices   
4       P005     Wireless Speaker            Audio   

                                         Description  Stock        Seasons  \
0  C

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Email: Subscription cancellation Hello, I have a question regarding subscription cancellation. Please assist.
Response: Subject: Re: Subscription Cancellation

Dear valued customer,

Thank you for reaching out to us regarding your subscription cancellation inquiry. However, after retrieving the relevant information from our database, I'm afraid we don't have any products associated with a subscription service.

If you could provide more context or clarify which product or service you are referring to, I'll be more than happy to assist you further.

Best regards,
[Your Name]
Customer Support Assistant

Email: Need info on Fitness Band Pro Hello, I am interested in Fitness Band Pro. Could you let me know if it's available and the price?
Response: Subject: Re: Fitness Band Pro Availability and Price

Dear [Customer],

Thank you for your interest in the Fitness Band Pro. I'm happy to inform you that it is available in our database.

The Fitness Band Pro is an advanced fitness band with GP

In [ ]:
print(products['Name'].unique())